# EXP013 E03 CONV — corrected production run

Select **A100** (high host RAM preferred), then run cells in order and upload the matching **E03 CONV input bundle**. Section 3 starts a retained, real MSL graph epoch; section 4 validates the corrected roles and measures the unchanged baselines; section 5 runs all **22 fits** and the original scientific stages. Supply annotations only when section 6 requests them after a verified score seal. Automatic Drive backups and resume are inherited from E02.

This is a separate corrected run: **do not copy native-convolution E01/P01/E02 checkpoints or gates into E03**. Keep the same `RUN_SLOT` when resuming this E03 run. All dimensions, seeds, losses, batch sizes, splits, FP32 and reproducibility settings are frozen. Old scientific settings, inputs and evaluator are retained.

Your preceding graph benchmark reported numerical/deterministic PASS and **501.39×** speedup from replacing `SlowDilated2d` dispatch with the equivalent FP32 convolution algebra. This build has CPU regression evidence; new A100 receipts are generated here. Removed/fusion and GDN/CATCH performance remain unmeasured until section 4. 40GB fit is measured, never assumed.


## 1. Connect Drive and retrieve the exact E03 bundle

Authorise the mount once. Drive stores backups and archives; training stays on local `/content`. Keep one notebook/VM per run folder.

In [ ]:
import os, sys, json, hashlib, zipfile, shutil, uuid
from pathlib import Path
from google.colab import drive, files
drive.mount('/content/drive')
RUN_SLOT = 'main'  # Keep unchanged when resuming. A new slot is a separate run.
if not RUN_SLOT.replace('_','').replace('-','').isalnum(): raise ValueError('Use a simple run slot name')
VERSION = 'EXP013_COMMAND_REFERENCE_V1_E03_CONV'
DRIVE = Path('/content/drive/MyDrive')/VERSION/RUN_SLOT
DRIVE.mkdir(parents=True,exist_ok=True)
WORK = Path('/content')/('EXP013_E03_work_'+RUN_SLOT)
CACHE = Path('/content')/('EXP013_E03_cache_'+RUN_SLOT); CACHE.mkdir(exist_ok=True)
PACKAGE = Path('/content/EXP013_E03_package')
BUNDLE_NAME = VERSION+'_input_bundle.zip'
BUNDLE_SHA256 = '4269c3c0e3ac446f3d8b52deb2fc4d21c1a2252afe0f1f557e82ee5e2a3ab2db'
MANIFEST_SHA256 = '772784ed57788cc9f4aa87d07d4bcb03bbf2bd03b382063a8b693090c2282045'
def file_sha(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda:f.read(2**20),b''):h.update(chunk)
    return h.hexdigest()
bundle=Path('/content')/BUNDLE_NAME
cached=DRIVE/BUNDLE_NAME
if not bundle.exists():
    if cached.exists():
        pending=bundle.with_name(bundle.name+'.partial-'+uuid.uuid4().hex)
        shutil.copyfile(cached,pending)
        if file_sha(pending)!=BUNDLE_SHA256:raise RuntimeError('Drive cached bundle checksum differs')
        os.replace(pending,bundle)
    else:
        os.chdir('/content'); print('Upload only '+BUNDLE_NAME+'. No annotation CSV here.',flush=True)
        uploaded=files.upload()
        if set(uploaded)!={BUNDLE_NAME}: raise RuntimeError('Expected exactly the E03 bundle')
        del uploaded
if file_sha(bundle)!=BUNDLE_SHA256: raise RuntimeError('Bundle checksum differs')
with zipfile.ZipFile(bundle) as z:
    names=z.namelist()
    if len(names)!=len(set(names)):raise RuntimeError('Duplicate ZIP members')
    mb=z.read('PACKAGE_MANIFEST.json')
    if hashlib.sha256(mb).hexdigest()!=MANIFEST_SHA256:raise RuntimeError('Package manifest differs')
    expected=json.loads(mb)['files']
    if set(names)!={*expected,'PACKAGE_MANIFEST.json'}:raise RuntimeError('Bundle roster differs')
    for info in z.infolist():
        p=Path(info.filename)
        if p.is_absolute() or '..' in p.parts or (info.external_attr>>16)&0o170000==0o120000:raise RuntimeError('Unsafe ZIP member')
    if not PACKAGE.exists():
        temporary=PACKAGE.with_name(PACKAGE.name+'.unpack-'+uuid.uuid4().hex); temporary.mkdir()
        for name in names:
            payload=z.read(name)
            if name in expected and hashlib.sha256(payload).hexdigest()!=expected[name]:raise RuntimeError('Payload differs')
            p=temporary/name;p.parent.mkdir(parents=True,exist_ok=True);p.write_bytes(payload)
        os.replace(temporary,PACKAGE)
actual={p.relative_to(PACKAGE).as_posix() for p in PACKAGE.rglob('*') if p.is_file() and '__pycache__' not in p.parts}
if actual!={*expected,'PACKAGE_MANIFEST.json'}:raise RuntimeError('Local package roster differs')
if file_sha(PACKAGE/'PACKAGE_MANIFEST.json')!=MANIFEST_SHA256:raise RuntimeError('Local manifest differs')
for name,h in expected.items():
    if file_sha(PACKAGE/name)!=h:raise RuntimeError('Local source changed: '+name)
if not cached.exists():
    pending=cached.with_name(cached.name+'.partial-'+uuid.uuid4().hex)
    with bundle.open('rb') as src,pending.open('wb') as dst:
        shutil.copyfileobj(src,dst,8*2**20);dst.flush();os.fsync(dst.fileno())
    if file_sha(pending)!=BUNDLE_SHA256:raise RuntimeError('Drive bundle copy differs')
    os.replace(pending,cached)
elif file_sha(cached)!=BUNDLE_SHA256:raise RuntimeError('Drive bundle cache differs; preserve and inspect')
print('Verified E03 package. Dedicated recovery path:',DRIVE,flush=True)

## 2. Install the frozen package versions and prepare the worker

Scientific imports run in fresh child processes. No old GPU gate or runtime identity is rewritten.

In [ ]:
import subprocess, importlib.metadata as md
if sys.version_info[:2] not in [(3,12),(3,13)]:raise RuntimeError('Use the frozen Python 3.12/3.13 Colab setup')
os.environ.update(CUBLAS_WORKSPACE_CONFIG=':4096:8',OMP_NUM_THREADS='1',MKL_NUM_THREADS='1',OPENBLAS_NUM_THREADS='1')
def installed(name):
    try:return md.version(name)
    except md.PackageNotFoundError:return None
extensions=[n for n in ['torch-scatter','torch-sparse','pyg-lib','torch-cluster','torch-spline-conv'] if installed(n)]
if extensions:subprocess.run([sys.executable,'-m','pip','uninstall','-y',*extensions],check=True)
if installed('torch')!='2.6.0+cu126':
    subprocess.run([sys.executable,'-m','pip','install','torch==2.6.0','--index-url','https://download.pytorch.org/whl/cu126'],check=True)
pins={'numpy':'2.1.3','pandas':'2.2.3','scipy':'1.15.2','scikit-learn':'1.6.1','einops':'0.8.1','torch-geometric':'2.6.1','matplotlib':'3.10.1'}
needed=[f'{n}=={v}' for n,v in pins.items() if installed(n)!=v]
if needed:subprocess.run([sys.executable,'-m','pip','install',*needed],check=True)
if installed('torch')!='2.6.0+cu126' or any(installed(n)!=v for n,v in pins.items()):raise RuntimeError('Frozen dependency pins differ')
sys.path.insert(0,str(PACKAGE/'src'))
from simple_launch import launch
def run(action, total_timeout=None, labels=None):
    cmd=[sys.executable,'-u',str(PACKAGE/'src/simple_run.py'),action,'--work',str(WORK),'--drive',str(DRIVE),'--cache',str(CACHE),'--device','cuda','--parent-pid',str(os.getpid())]
    if labels is not None:cmd+=['--labels',str(labels)]
    try:launch(cmd,CACHE,total_timeout=total_timeout)
    except BaseException:
        # Immutable diagnostic copies only; never snapshot an active scientific writer.
        diagnostics=DRIVE/'diagnostics'/uuid.uuid4().hex;diagnostics.mkdir(parents=True)
        for name in ['operation.json','progress.jsonl','worker_stdout.log']:
            src=CACHE/name
            if src.exists():
                dst=diagnostics/name;shutil.copyfile(src,dst)
                if file_sha(src)!=file_sha(dst):print('Diagnostic copy changed while copying:',name,flush=True)
        print('Execution stopped. Preserve diagnostics:',diagnostics,flush=True)
        raise
run('startup',total_timeout=1200)

## 3. First real model: complete epoch and exact checkpoint restore

This runs **MSL graph seed 17**, with all **127 FIT and four STOP batches**, batch 256, original loss/optimizer and FP32 settings. The first epoch and the next update are kept for this fit. No selection, confirmation or TEST access is authorised here.

Compute operations have a 180-second deadline; the complete epoch has a 1,200-second bound. This cell has a 1,800-second overall bound. A timeout stops and retains the last verified checkpoint; it does not shorten an epoch or grant a gate.

In [ ]:
run('smoke',total_timeout=1800)
record=json.loads((WORK/'SMOKE_GATE.json').read_text())
print('Real epoch seconds:',round(record['epoch1']['seconds'],2))
print('Checkpoint restore:',record['checkpoint_restore_replay'])
print('Retained continuation:',record['first_fit_continues_from'])

## 4. Corrected-role equivalence and remaining full-batch feasibility

The existing hardware action now compares **two complete Adam updates** against frozen native convolution for all six graph/removal/fusion mission combinations at **batch 256**, then requires exact corrected RNG/Adam replay. These private updates are discarded. This adds approximately several minutes of slow native reference computation, once per E03 identity; completed role checks are backed up and reused. It is a bounded correctness check, not a profiler or sweep.

FP32 sums differ: the CPU fusion trial reached **2.73e-6 parameter error**. E03 explicitly uses output/loss atol2e-6/rtol2e-5, gradient2e-7/2e-4, parameter5e-6/2e-5, Adam moment2e-8/2e-4; every update is also checked against an independent Adam equation calculation. RNG, injections, counters and corrected replay must be exact. The full audit and original failed tolerance trial are included in the bundle. Tolerances are fixed before GPU execution.

The nine remaining E02 feasibility probes keep native scientific batches **256 / 32 / 128** for new models / GDN / CATCH. They test gradients, optimizer memory, native scores and checkpoint restoration. MSL CATCH has **211 million parameters**. GDN/CATCH remain unchanged. An OOM, regression failure or insufficient measured memory margin stops; settings are never reduced. Each numerical comparison/probe is bounded to 600 seconds.


In [ ]:
run('hardware')
gate=json.loads((WORK/'ENGINEERING_GATE.json').read_text())
hours=0.0
for r in gate['full_batches']:
    copies=3 if r['role'] in ['graph','removed','fusion'] else 1
    hours+=copies*r['epoch_cap_training_seconds_projection']/3600
    peak=r.get('peak_allocated_bytes')
    print(r['mission'],r['role'],'batch',r['batch_size'],'peak allocated GiB',None if peak is None else round(peak/2**30,3),'epoch-cap projection h',round(r['epoch_cap_training_seconds_projection']/3600,3))
    numerical=r.get('convolution_regression')
    if numerical:
        for kind in ['outputs','gradients','parameters','adam']:
            maximum=max(step['comparisons'][kind]['max_absolute_error'] for step in numerical['native_vs_corrected'])
            print(' ',kind,'native/corrected max absolute difference:',maximum)
        print('  Exact corrected replay:',numerical['corrected_replay_bitwise_equal'])
print('Measured-batch extrapolation for all 22 training/STOP epoch caps:',round(hours,2),'hours.')
print('Selection, confirmation, TEST scoring and Drive transfers add time; early stopping can reduce fit time.')
print('This is a projection from one full real graph epoch and representative remaining batches, not an end-to-end guarantee.')


In [ ]:
from pathlib import Path
import os, sys, hashlib
from google.colab import files

uploaded = files.upload()
name = 'EXP013_E03_P01_PARAMETER_GATE.py'
if set(uploaded) != {name}:
    raise RuntimeError('Upload only the P01 Python amendment')

p = Path('/content') / name
p.write_bytes(uploaded[name])
del uploaded

assert hashlib.sha256(p.read_bytes()).hexdigest() == 'fb4522f3926c09d9b52b46a2dd776e79608b505b7081b513832d4a9020d63eea'

os.environ['EXP013_E03_PACKAGE'] = str(PACKAGE)

launch(
    [sys.executable, '-u', str(p), 'hardware',
     '--work', str(WORK),
     '--drive', str(DRIVE),
     '--cache', str(CACHE),
     '--device', 'cuda',
     '--parent-pid', str(os.getpid())],
    CACHE,
    total_timeout=1800
)

run('develop')

In [ ]:
from pathlib import Path
import ast

log = Path('/content/EXP013_E03_cache_main/worker_stdout.log').read_text()
marker = 'Full-batch native/corrected FP32 regression failed: '
result = ast.literal_eval(log.rsplit(marker, 1)[1].splitlines()[0])

for i, update in enumerate(result, 1):
    for name, check in update['comparisons'].items():
        if not check['pass']:
            print('Update:', i, '| Failed:', name)
            print('Maximum absolute error:', check['max_absolute_error'])
            print('Failed tensors:', check['failures'][:20])

## 5. Continue all 22 fits, select, freeze, confirm and seal

The first fit continues from its verified checkpoint. Valid completed fits and committed score calls are reused. Automatic safe-boundary backups are due every 600 seconds and forced at fit/stage completion. All scientific work stays local. A strict 15-minute wall-clock recovery window cannot be guaranteed for slow copies or long scoring calls.

In [ ]:
run('develop')

## 6. Verify the score seal, then supply annotations and finish

Only this cell may look up or request the NASA annotation CSV. It verifies the complete score seal first and the evaluator verifies it again before parsing. The post-seal cache is never inspected during startup or training.

In [ ]:
run('verify-seal')
annotation=Path('/content/NASA_annotations_after_verified_seal.csv')
annotation_cache=DRIVE/'NASA_annotations_after_verified_seal.csv'
if not annotation.exists():
    if annotation_cache.exists():shutil.copyfile(annotation_cache,annotation)
    else:
        os.chdir('/content');print('Score seal verified. Upload the original labeled_anomalies.csv now.',flush=True)
        uploaded=files.upload()
        if set(uploaded)!={'labeled_anomalies.csv'}:raise RuntimeError('Expected the original annotation CSV only')
        shutil.copyfile('/content/labeled_anomalies.csv',annotation);del uploaded
expected_annotation_sha='057ce2d6c8875982bf4e5404aefea14efdcbce413d80826d2b737c95b59b7539'
if file_sha(annotation)!=expected_annotation_sha:raise RuntimeError('Wrong annotation bytes')
if not annotation_cache.exists():
    pending=annotation_cache.with_name(annotation_cache.name+'.partial-'+uuid.uuid4().hex)
    shutil.copyfile(annotation,pending)
    if file_sha(pending)!=expected_annotation_sha:raise RuntimeError('Annotation cache checksum differs')
    os.replace(pending,annotation_cache)
run('evaluate',labels=annotation)
print('Final report, audit and recovery states:',DRIVE)
print('Final downloadable results archive:',DRIVE/'FINAL_RESULTS.zip')
from IPython.display import display, Markdown
display(Markdown((WORK/'reports/RESEARCH_REPORT.md').read_text()))